In [1]:
!pip install atoti psycopg2-binary sqlalchemy pandas


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import atoti as tt

from sqlalchemy import create_engine
from urllib.parse import quote_plus

Welcome to Atoti 0.9.14!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


# Connect postgreSQL

In [2]:
from sqlalchemy import create_engine

username = "postgres"
password = "adistiebil123"
host = "localhost"
port = "5432"
database = "dwh_air_quality"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# Dimensi Data

In [3]:
fact = pd.read_sql(
    """
    SELECT *
    FROM fact_air_quality_health
    """,
    engine
)

dim_time = pd.read_sql(
    """
    SELECT *
    FROM dim_time
    """,
    engine
)

dim_location = pd.read_sql(
    """
    SELECT *
    FROM dim_location
    """,
    engine
)

dim_indicator = pd.read_sql(
    """
    SELECT *
    FROM dim_indicator
    """,
    engine
)

dim_batch = pd.read_sql(
    """
    SELECT *
    FROM dim_batch
    """,
    engine
)

print("Fact Shape :", fact.shape)
print("Time Shape :", dim_time.shape)
print("Location Shape :", dim_location.shape)
print("Indicator Shape :", dim_indicator.shape)
print("Batch Shape :", dim_batch.shape)

Fact Shape : (18862, 7)
Time Shape : (57, 6)
Location Shape : (141, 4)
Indicator Shape : (21, 4)
Batch Shape : (3, 6)


In [4]:
print(fact.dtypes)
print(dim_time.dtypes)
print(dim_location.dtypes)
print(dim_indicator.dtypes)
print(dim_batch.dtypes)

fact_id            int64
unique_id          int64
time_id            int64
location_id        int64
indicator_key      int64
batch_id           int64
data_value       float64
dtype: object
time_id                 int64
start_date     datetime64[ns]
year                    int64
month                   int64
quarter                 int64
time_period            object
dtype: object
location_id        int64
geo_type_name     object
geo_join_id        int64
geo_place_name    object
dtype: object
indicator_key           int64
source_indicator_id     int64
indicator_name         object
indicator_category     object
dtype: object
batch_id        int64
batch_name     object
start_year      int64
end_year        int64
total_rows      int64
description    object
dtype: object


# Fact Table

In [5]:
fact["data_value"] = pd.to_numeric(
    fact["data_value"],
    errors="coerce"
)

fact["time_id"] = pd.to_numeric(
    fact["time_id"],
    errors="coerce"
)

fact["location_id"] = pd.to_numeric(
    fact["location_id"],
    errors="coerce"
)

fact["indicator_key"] = pd.to_numeric(
    fact["indicator_key"],
    errors="coerce"
)

fact["batch_id"] = pd.to_numeric(
    fact["batch_id"],
    errors="coerce"
)

fact = fact.dropna(
    subset=[
        "data_value",
        "time_id",
        "location_id",
        "indicator_key",
        "batch_id"
    ]
)

print("Shape setelah cleaning :", fact.shape)

Shape setelah cleaning : (18862, 7)


In [6]:
fact["time_id"] = fact["time_id"].astype(str)
fact["location_id"] = fact["location_id"].astype(str)
fact["indicator_key"] = fact["indicator_key"].astype(str)
fact["batch_id"] = fact["batch_id"].astype(str)

dim_time["time_id"] = dim_time["time_id"].astype(str)
dim_location["location_id"] = dim_location["location_id"].astype(str)
dim_indicator["indicator_key"] = dim_indicator["indicator_key"].astype(str)
dim_batch["batch_id"] = dim_batch["batch_id"].astype(str)

print("Key berhasil diubah ke string")

Key berhasil diubah ke string


In [7]:
dim_time["year"] = dim_time["year"].astype(str)
dim_time["month"] = dim_time["month"].astype(str)
dim_time["quarter"] = dim_time["quarter"].astype(str)
dim_time["time_period"] = dim_time["time_period"].astype(str)

dim_location["geo_type_name"] = dim_location["geo_type_name"].astype(str)
dim_location["geo_join_id"] = dim_location["geo_join_id"].astype(str)
dim_location["geo_place_name"] = dim_location["geo_place_name"].astype(str)

dim_indicator["source_indicator_id"] = dim_indicator["source_indicator_id"].astype(str)
dim_indicator["indicator_name"] = dim_indicator["indicator_name"].astype(str)
dim_indicator["indicator_category"] = dim_indicator["indicator_category"].astype(str)

dim_batch["batch_name"] = dim_batch["batch_name"].astype(str)
dim_batch["start_year"] = dim_batch["start_year"].astype(str)
dim_batch["end_year"] = dim_batch["end_year"].astype(str)
dim_batch["description"] = dim_batch["description"].astype(str)

print("Key join dan atribut hierarchy berhasil diubah")

print(dim_time.dtypes)
print(dim_location.dtypes)
print(dim_indicator.dtypes)
print(dim_batch.dtypes)

Key join dan atribut hierarchy berhasil diubah
time_id                object
start_date     datetime64[ns]
year                   object
month                  object
quarter                object
time_period            object
dtype: object
location_id       object
geo_type_name     object
geo_join_id       object
geo_place_name    object
dtype: object
indicator_key          object
source_indicator_id    object
indicator_name         object
indicator_category     object
dtype: object
batch_id       object
batch_name     object
start_year     object
end_year       object
total_rows      int64
description    object
dtype: object


# Atoti

In [8]:
session = tt.Session.start()

In [9]:
fact_table = session.read_pandas(
    fact,
    table_name="fact_air_quality_health",
    keys=["fact_id"]
)

time_table = session.read_pandas(
    dim_time,
    table_name="dim_time",
    keys=["time_id"]
)

location_table = session.read_pandas(
    dim_location,
    table_name="dim_location",
    keys=["location_id"]
)

indicator_table = session.read_pandas(
    dim_indicator,
    table_name="dim_indicator",
    keys=["indicator_key"]
)

batch_table = session.read_pandas(
    dim_batch,
    table_name="dim_batch",
    keys=["batch_id"]
)

print("Semua tabel berhasil di-load ke Atoti")

Semua tabel berhasil di-load ke Atoti


In [10]:
fact_table.join(
    time_table,
    fact_table["time_id"] == time_table["time_id"]
)

fact_table.join(
    location_table,
    fact_table["location_id"] == location_table["location_id"]
)

fact_table.join(
    indicator_table,
    fact_table["indicator_key"] == indicator_table["indicator_key"]
)

fact_table.join(
    batch_table,
    fact_table["batch_id"] == batch_table["batch_id"]
)

print("Join antar tabel berhasil")

Join antar tabel berhasil


In [11]:
cube = session.create_cube(
    fact_table,
    name="Air Quality Health Cube"
)

print(cube)

In [12]:
m = cube.measures

m["Total Data Value"] = tt.agg.sum(
    fact_table["data_value"]
)

m["Average Data Value"] = tt.agg.mean(
    fact_table["data_value"]
)

m["Maximum Data Value"] = tt.agg.max(
    fact_table["data_value"]
)

m["Minimum Data Value"] = tt.agg.min(
    fact_table["data_value"]
)

print("Measures berhasil dibuat")
print(cube.measures)

Measures berhasil dibuat
{'Average Data Value': <atoti.measure.Measure object at 0x0000018CBF284560>, 'Maximum Data Value': <atoti.measure.Measure object at 0x0000018CDBAF8AD0>, 'Minimum Data Value': <atoti.measure.Measure object at 0x0000018CDBAF8E90>, 'Total Data Value': <atoti.measure.Measure object at 0x0000018CDBAF9BB0>, 'contributors.COUNT': <atoti.measure.Measure object at 0x0000018CDBAF9E80>, 'data_value.MEAN': <atoti.measure.Measure object at 0x0000018CDBAFA690>, 'data_value.SUM': <atoti.measure.Measure object at 0x0000018CDB8AAB10>, 'unique_id.MEAN': <atoti.measure.Measure object at 0x0000018CDB8A87D0>, 'unique_id.SUM': <atoti.measure.Measure object at 0x0000018CDB8A87A0>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x0000018CDB8AA8A0>}


In [13]:
print(cube.measures)

{'Average Data Value': <atoti.measure.Measure object at 0x0000018CDB8D2120>, 'Maximum Data Value': <atoti.measure.Measure object at 0x0000018CDB8D1FD0>, 'Minimum Data Value': <atoti.measure.Measure object at 0x0000018CDB8D1A00>, 'Total Data Value': <atoti.measure.Measure object at 0x0000018CDBB053D0>, 'contributors.COUNT': <atoti.measure.Measure object at 0x0000018CDBB051F0>, 'data_value.MEAN': <atoti.measure.Measure object at 0x0000018CDBB05310>, 'data_value.SUM': <atoti.measure.Measure object at 0x0000018CDBB04A70>, 'unique_id.MEAN': <atoti.measure.Measure object at 0x0000018CDBB04EC0>, 'unique_id.SUM': <atoti.measure.Measure object at 0x0000018CDBB05160>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x0000018CDBB04B00>}


In [14]:
try:
    print(session.url)
except:
    print(session.link())

http://localhost:52614
